# Approach 2 · A4 partial DINOv2 fine-tuning

A4 is a separate follow-up to the A3 frozen model in Matched Selection. The main comparison is **A3 raw versus A4 raw**. The development split, augmentations, losses, heads, initialization scheme, FP32 mode and epoch budget follow the A3 setup. A controlled comparison of backbone fine-tuning also requires the same full-training refit procedure for both. A4 adds gradients to DINOv2's last transformer block and final LayerNorm. The remaining backbone parameters stay frozen.

Each Setup creates an independent run folder and initializes A4 from the original pretrained DINOv2 weights.

## 1. Setup

Place this notebook in the project and use the project Jupyter kernel. Set `DEBUG=True` for the short training check, or `DEBUG=False` for A4 development, full-training refit and official evaluation.

In [ ]:
DEBUG = True

from datetime import datetime, timezone
from pathlib import Path
import copy
import hashlib
import json
import sys

import pandas as pd
import timm
import torch
from IPython.display import display
from PIL import Image
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.transforms import functional as TF

here = Path.cwd()
ROOT = next((p for p in (here, *here.parents) if (p / "common" / "dataset.py").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Open the notebook inside the team repository.")
sys.path.insert(0, str(ROOT))

from common import config as team
from common.dataset import PlantDataset, get_loaders, get_pairs, get_transforms, load_split
from common.evaluate import PERTURBATIONS, harmonic_mean, save_confusion, top_k_accuracy

BACKBONE = "vit_base_patch14_dinov2.lvd142m"
REFERENCE_COMMIT = "0fece2daff4917431bb2a5b16b65718ab100d05c"
DEVICE = team.DEVICE
NUM_CROPS, NUM_DISEASES = team.NUM_CROPS, team.NUM_DISEASES
team.NUM_WORKERS = 0
BATCH_SIZE, EPOCHS = (8, 1) if DEBUG else (16, 20)
HEAD_LR, BACKBONE_LR, WEIGHT_DECAY = 1e-4, 1e-5, 1e-4
SMOOTHING = 0.1
DECOR_WEIGHT, SUPCON_WEIGHT, TEMPERATURE = 0.05, 0.10, 0.10
BLUR_PROB, BLUR_SIGMA = 0.15, 0.7
GAMMAS = [step / 2 for step in range(25)]
RUN_NAME = "approach2_finetune_a4_full_refit" + ("_debug" if DEBUG else "")
RECIPE = "A4_Last_Block_FineTune"
RUN_ID = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S%fZ")
RESULTS_DIR = team.RESULTS_DIR / RUN_NAME / RUN_ID
DEVELOPMENT_DIR = RESULTS_DIR / "development"
FINAL_DIR = RESULTS_DIR / "final"
OFFICIAL_DIR = RESULTS_DIR / "official" / "pair_identity_order"
CHECKPOINT = team.CHECKPOINT_DIR / RUN_NAME / RUN_ID / "model.pt"
LOCK = FINAL_DIR / "selection_lock.json"
REFERENCE_REPORT = (team.RESULTS_DIR / "approach2_baseline2_selected" /
                    "official" / "pair_identity_order" / "official_report.json")

if not team.TRAIN_CSV.is_file() or not team.IMAGE_DIR.is_dir():
    raise FileNotFoundError("Training CSV or image directory is missing.")
if team.NUM_PAIRS != team.NUM_SEEN_PAIRS + team.NUM_UNSEEN_PAIRS:
    raise ValueError("Use the shared 323/333 evaluation API.")

## 2. Data

Use the shared Baseline 2 training and seen-validation split. Hold out ten complete training-derived compositions during development. Keep each constituent crop and disease represented in development fit. Five proxy pairs and seen validation images form tune, and another five pairs and disjoint seen images form the check. The full refit uses every row of the training CSV, including the training-derived proxy pairs and seen-validation images.

In [ ]:
def image_key(value):
    result = Path(str(value).strip().replace("\\", "/"))
    if not result.name or result.is_absolute() or ".." in result.parts or ":" in str(result):
        raise ValueError(f"Expected a relative image filename, got {value!r}.")
    return result.as_posix()


def sha256_file(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def atomic_json(path, value):
    temp = path.with_suffix(".json.tmp")
    temp.write_text(json.dumps(value, indent=2), encoding="utf-8")
    temp.replace(path)


def atomic_save(path, value):
    temp = path.with_suffix(".pt.tmp")
    torch.save(value, temp)
    temp.replace(path)


def calibration_split(fit_frame, val_frame):
    """Hold out complete training pairs from the development heads only."""
    grouped = fit_frame.groupby("pair_row", sort=True)
    rows = grouped.agg(n=("image", "size"), crop=("crop", "first"), disease=("disease", "first"))
    crops = rows.crop.value_counts().to_dict()
    diseases = rows.disease.value_counts().to_dict()
    eligible = rows[rows.n.ge(5)].sample(frac=1, random_state=team.SEED + 1)
    held = []
    for pair_row, row in eligible.iterrows():
        if crops[row.crop] > 1 and diseases[row.disease] > 1:
            held.append(int(pair_row))
            crops[row.crop] -= 1
            diseases[row.disease] -= 1
        if len(held) == 10:
            break
    if len(held) < 10:
        raise ValueError("Ten train-derived holdouts with fit support were not available.")
    seen = val_frame.loc[~val_frame.pair_row.isin(held)].sample(frac=1, random_state=team.SEED + 2)
    cut = len(seen) // 2
    frames = {
        "tune_seen": seen.iloc[:cut],
        "tune_proxy": train_full.loc[train_full.pair_row.isin(held[:5])],
        "check_seen": seen.iloc[cut:],
        "check_proxy": train_full.loc[train_full.pair_row.isin(held[5:])],
    }
    if any(group.empty for group in frames.values()):
        raise ValueError("Calibration needs both seen and proxy images in each group.")
    fit_mask = torch.ones(len(train_pairs), dtype=torch.bool)
    fit_mask[held] = False
    return fit_mask, frames


train_full = load_split(team.TRAIN_CSV)
train_full["image"] = train_full.image.map(image_key)
if not train_full.image.is_unique:
    raise ValueError("Duplicate training images.")
train_pairs = get_pairs()
if train_pairs.shape != (team.NUM_SEEN_PAIRS, 2):
    raise ValueError("Unexpected training pair table.")
_, image_transform = get_transforms("imagenet")

settings = {
    "reference_commit": REFERENCE_COMMIT, "backbone": BACKBONE, "seed": team.SEED,
    "image_size": team.IMAGE_SIZE, "resize_size": team.RESIZE_SIZE, "val_fraction": team.VAL_FRACTION,
    "batch_size": BATCH_SIZE, "epochs": EPOCHS, "head_lr": HEAD_LR, "backbone_lr": BACKBONE_LR,
    "weight_decay": WEIGHT_DECAY, "optimizer": "AdamW", "schedule": "CosineAnnealingLR",
    "label_smoothing": SMOOTHING, "trainable_backbone": ["blocks.11", "norm"],
    "decor_weight": DECOR_WEIGHT, "supcon_weight": SUPCON_WEIGHT, "temperature": TEMPERATURE,
    "blur_probability": BLUR_PROB, "blur_sigma": BLUR_SIGMA,
    "calibration_grid": GAMMAS, "train_csv_sha256": sha256_file(team.TRAIN_CSV),
    "torch": str(torch.__version__), "timm": timm.__version__, "debug": DEBUG,
}

## 3. Model

A4 fine-tunes the last block, final LayerNorm and two Linear Heads on a frozen DINOv2 prefix. The code checks that the split forward equals the original timm forward before training and after export. The A3 crop and disease CE, Logit Decorrelation and Cross-crop SupCon losses stay in place, and A3's mild Gaussian Blur is applied at the same probability. Use AdamW with head LR 1e-4 and last block plus LayerNorm LR 1e-5, both with a cosine schedule. The extra trainable tail tests feature adaptation while retaining the same inference graph.

In [ ]:
class MultiTaskNet(nn.Module):
    """Latest Baseline 2 architecture."""
    def __init__(self, pretrained):
        super().__init__()
        self.backbone = timm.create_model(BACKBONE, pretrained=pretrained, num_classes=0,
                                          img_size=team.IMAGE_SIZE)
        self.backbone.requires_grad_(False)
        self.crop_head = nn.Linear(self.backbone.num_features, NUM_CROPS)
        self.disease_head = nn.Linear(self.backbone.num_features, NUM_DISEASES)

    def forward(self, images):
        features = self.backbone(images)
        return self.crop_head(features), self.disease_head(features)


def pair_scores(crop_logits, disease_logits, pairs):
    pairs = pairs.to(crop_logits.device)
    crop_log = F.log_softmax(crop_logits.float(), dim=1)
    disease_log = F.log_softmax(disease_logits.float(), dim=1)
    return crop_log[:, pairs[:, 0]] + disease_log[:, pairs[:, 1]]


def center_logits(logits):
    # Remove the per-image offset to which categorical predictions are invariant.
    return logits - logits.mean(dim=1, keepdim=True)


def decorrelation(crop, disease):
    def standardize(x):
        x = x - x.mean(0)
        return x / (x.square().mean(0) + 1e-4).sqrt()

    cross_correlation = standardize(crop).T @ standardize(disease) / len(crop)
    return cross_correlation.square().mean()


def cross_crop_supcon(embedding, crop, disease):
    z = F.normalize(embedding, dim=1)
    same_disease = disease[:, None].eq(disease[None, :])
    positives = same_disease & crop[:, None].ne(crop[None, :])
    negatives = ~same_disease
    similarity = z @ z.T / TEMPERATURE
    normalizer = similarity.masked_fill(~(positives | negatives), -1e9).logsumexp(dim=1)
    counts = positives.sum(dim=1)
    positive_mean = similarity.masked_fill(~positives, 0.0).sum(dim=1) / counts.clamp_min(1)
    usable = (counts > 0) & negatives.any(dim=1)
    per_anchor = (normalizer - positive_mean).masked_fill(~usable, 0.0)
    return per_anchor.sum() / usable.sum().clamp_min(1), usable.sum()


class FineTuneTail(nn.Module):
    """Independent trainable tail and heads on the shared frozen DINOv2 prefix."""
    def __init__(self, model):
        super().__init__()
        self.block = copy.deepcopy(model.backbone.blocks[-1]).requires_grad_(True)
        self.norm = copy.deepcopy(model.backbone.norm).requires_grad_(True)
        self.crop_head = copy.deepcopy(model.crop_head)
        self.disease_head = copy.deepcopy(model.disease_head)

    def forward(self, tokens, backbone):
        tokens = self.norm(self.block(tokens))
        features = backbone.forward_head(tokens)
        return self.crop_head(features), self.disease_head(features)


@torch.no_grad()
def frozen_prefix(backbone, images):
    tokens = backbone.patch_embed(images)
    tokens = backbone._pos_embed(tokens)
    tokens = backbone.norm_pre(backbone.patch_drop(tokens))
    for block in backbone.blocks[:-1]:
        tokens = block(tokens)
    return tokens


def augment_blur(images, generator):
    selected = torch.rand(len(images), generator=generator) < BLUR_PROB
    if not selected.any():
        return images
    selected = selected.to(images.device)
    augmented = images.clone()
    augmented[selected] = TF.gaussian_blur(images[selected], kernel_size=[5, 5],
                                           sigma=[BLUR_SIGMA, BLUR_SIGMA])
    return augmented


@torch.no_grad()
def verify_prefix(model, tail, images):
    # Check the lightweight split against the standard timm forward before training.
    from_prefix = tail(frozen_prefix(model.backbone, images), model.backbone)
    full = model(images)
    for a, b in zip(from_prefix, full):
        torch.testing.assert_close(a, b, rtol=1e-5, atol=1e-6)

## 4. Development and epoch selection

Select the epoch count using raw seen/proxy-unseen Top-1 harmonic mean on tune. Break ties by proxy-unseen Top-1, seen Top-1 and earlier epoch. Tune gamma on the selected development weights and report the separate check. Carry the chosen epoch count and gamma into the final stage.

In [ ]:
def calibration_metrics(seen, proxy, fit_mask, gamma):
    seen_scores, seen_y = seen
    proxy_scores, proxy_y = proxy
    penalty = gamma * fit_mask
    s = top_k_accuracy(seen_scores - penalty, seen_y, 1)
    u = top_k_accuracy(proxy_scores - penalty, proxy_y, 1)
    return {"gamma": gamma, "seen_top1": s, "proxy_top1": u, "harmonic_top1": harmonic_mean(s, u)}


def selection_key(row):
    return (row["harmonic_top1"], row["proxy_top1"], row["seen_top1"], -row["epoch"])


def make_optimizer(tail):
    heads = [*tail.crop_head.parameters(), *tail.disease_head.parameters()]
    backbone = [*tail.block.parameters(), *tail.norm.parameters()]
    groups = [{"params": heads, "lr": HEAD_LR}, {"params": backbone, "lr": BACKBONE_LR}]
    return torch.optim.AdamW(groups, weight_decay=WEIGHT_DECAY)


def snapshot_tail(tail):
    return {name: value.detach().cpu().clone() for name, value in tail.state_dict().items()}


def train_tail(tail, optimizer, criterion, tokens, backbone, crop, disease):
    crop_logits, disease_logits = tail(tokens, backbone)
    crop_repr, disease_repr = center_logits(crop_logits), center_logits(disease_logits)
    loss = criterion(crop_logits, crop) + criterion(disease_logits, disease)
    contrast, anchors = cross_crop_supcon(disease_repr, crop, disease)
    loss = loss + DECOR_WEIGHT * decorrelation(crop_repr, disease_repr) + SUPCON_WEIGHT * contrast
    if not torch.isfinite(loss):
        raise FloatingPointError("Non-finite fine-tuning loss.")
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()
    return loss.item(), int(anchors.detach())


@torch.no_grad()
def cached_group(backbone, frame):
    rng = torch.Generator().manual_seed(team.SEED)
    loader = DataLoader(PlantDataset(frame, image_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=False, num_workers=0, generator=rng)
    return [(frozen_prefix(backbone, images.to(DEVICE)).cpu(), target.cpu())
            for images, _, _, target in loader]


@torch.no_grad()
def score_group(tail, backbone, batches):
    scores, labels = [], []
    for tokens, targets in batches:
        crop, disease = tail(tokens.to(DEVICE), backbone)
        scores.append(pair_scores(crop, disease, train_pairs).cpu())
        labels.append(targets)
    return torch.cat(scores), torch.cat(labels)


def choose_gamma(backbone, tail, fit_mask, frames, tune_cache):
    tune = [score_group(tail, backbone, tune_cache[key]) for key in ("tune_seen", "tune_proxy")]
    grid = [calibration_metrics(*tune, fit_mask, gamma) for gamma in GAMMAS]
    choice = max(grid, key=lambda row: (row["harmonic_top1"], row["proxy_top1"],
                                        row["seen_top1"], -row["gamma"]))
    gamma = choice["gamma"]
    check = [score_group(tail, backbone, cached_group(backbone, frames[key]))
             for key in ("check_seen", "check_proxy")]
    check_rows = [{"scoring": name, **calibration_metrics(*check, fit_mask, value)}
                  for name, value in (("raw", 0.0), ("calibrated", gamma))]
    pd.DataFrame(grid).to_csv(DEVELOPMENT_DIR / "calibration_tune_grid.csv", index=False)
    pd.DataFrame(check_rows).to_csv(DEVELOPMENT_DIR / "calibration_check.csv", index=False)
    return gamma


def read_final_lock():
    if not (LOCK.is_file() and CHECKPOINT.is_file()):
        raise FileNotFoundError("Missing A4 checkpoint or final selection lock.")
    saved = json.loads(LOCK.read_text(encoding="utf-8"))
    if saved["settings"] != settings or saved["checkpoint_sha256"] != sha256_file(CHECKPOINT):
        raise RuntimeError("Existing A4 run has different settings or checkpoint bytes.")
    if saved["seen_pairs"] != train_pairs.tolist():
        raise RuntimeError("Training candidate identities changed after the checkpoint was locked.")
    return saved


def make_training_model():
    model = MultiTaskNet(pretrained=True).to(DEVICE).eval()
    if len(model.backbone.blocks) != 12:
        raise ValueError("Expected the 12-block DINOv2 ViT-B/14.")
    return model, FineTuneTail(model).to(DEVICE).eval()


def train_epoch(model, tail, optimizer, criterion, loader, blur_rng, stage, epoch, epochs, fit_mask=None):
    total_loss, examples, anchors = 0.0, 0, 0
    for step, (images, crops, diseases, pair_rows) in enumerate(loader, 1):
        images = augment_blur(images.to(DEVICE), blur_rng)
        crops, diseases = crops.to(DEVICE), diseases.to(DEVICE)
        if fit_mask is not None:
            keep = fit_mask[pair_rows.to(DEVICE)]
            if not keep.any():
                continue
            images, crops, diseases = images[keep], crops[keep], diseases[keep]
        if epoch == 1 and examples == 0:
            verify_prefix(model, tail, images[:1])
        tokens = frozen_prefix(model.backbone, images)
        loss, count = train_tail(tail, optimizer, criterion, tokens, model.backbone, crops, diseases)
        total_loss += loss * len(images)
        examples += len(images)
        anchors += count
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"A4 {stage} epoch {epoch}/{epochs} · batch {step}/{len(loader)}", flush=True)
    return {"epoch": epoch, "images_trained": examples, "train_loss": total_loss / examples,
            "supcon_anchor_fraction": anchors / examples}


def run_development():
    DEVELOPMENT_DIR.mkdir(parents=True, exist_ok=True)
    team.set_seed()
    train, val, seen, unseen = get_loaders(BATCH_SIZE, norm="imagenet", debug=DEBUG)
    if seen is not None or unseen is not None:
        raise RuntimeError("Development loaders must never return official data.")
    train.pin_memory = False
    fit_mask, frames = (None, {}) if DEBUG else calibration_split(train.dataset.frame, val.dataset.frame)
    fit_device = fit_mask.to(DEVICE) if fit_mask is not None else None
    model, tail = make_training_model()
    optimizer = make_optimizer(tail)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    if not DEBUG:
        tune_cache = {key: cached_group(model.backbone, frames[key]) for key in ("tune_seen", "tune_proxy")}

    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)
    blur_rng = torch.Generator().manual_seed(team.SEED + 172)
    best, history = None, []
    for epoch in range(1, EPOCHS + 1):
        row = train_epoch(model, tail, optimizer, criterion, train, blur_rng,
                          "development", epoch, EPOCHS, fit_mask=fit_device)
        schedule.step()
        if not DEBUG:
            groups = [score_group(tail, model.backbone, tune_cache[key]) for key in ("tune_seen", "tune_proxy")]
            raw = calibration_metrics(*groups, fit_mask, gamma=0.0)
            row.update({key: value for key, value in raw.items() if key != "gamma"})
            if best is None or selection_key(row) > selection_key(best["scores"]):
                best = {"scores": row.copy(), "weights": snapshot_tail(tail)}
        history.append(row)
        print(row, flush=True)
    pd.DataFrame(history).to_csv(DEVELOPMENT_DIR / "training_history.csv", index=False)
    if DEBUG:
        return None

    tail.load_state_dict(best["weights"])
    gamma = choose_gamma(model.backbone, tail, fit_mask, frames, tune_cache)
    selection = {
        "selection": best["scores"], "gamma": gamma,
        "selection_tune_pair_rows": sorted(frames["tune_proxy"].pair_row.unique().tolist()),
        "calibration_check_pair_rows": sorted(frames["check_proxy"].pair_row.unique().tolist()),
    }
    atomic_json(DEVELOPMENT_DIR / "epoch_selection.json", selection)
    return selection


development = run_development()
if development is not None:
    display(pd.DataFrame([{**development["selection"], "locked_gamma": development["gamma"]}]))

## 5. Full-training refit

Initialize from the original pretrained DINOv2 weights with fresh heads. Train on every image in the training CSV for the selected epoch count using the development cosine schedule up to that epoch. Save the final weights and lock the checkpoint and the development-selected gamma before official evaluation. `final/refit_history.csv` records the number of images processed in each refit epoch.

In [ ]:
def refit_full_train(development):
    if LOCK.is_file():
        raise FileExistsError("This A4 run is complete. Rerun Setup to start again.")
    if sha256_file(team.TRAIN_CSV) != settings["train_csv_sha256"]:
        raise RuntimeError("Training CSV changed after development. Start a fresh run.")

    selected_epochs = int(development["selection"]["epoch"])
    if not 1 <= selected_epochs <= EPOCHS:
        raise ValueError("Selected refit length is outside the development epoch budget.")
    team.set_seed()
    model, tail = make_training_model()
    optimizer = make_optimizer(tail)
    schedule = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
    train_transform, _ = get_transforms("imagenet")
    loader = DataLoader(PlantDataset(train_full, train_transform, "imagenet"), batch_size=BATCH_SIZE,
                        shuffle=True, num_workers=team.NUM_WORKERS, drop_last=False,
                        generator=torch.Generator().manual_seed(team.SEED))
    criterion = nn.CrossEntropyLoss(label_smoothing=SMOOTHING)

    blur_rng = torch.Generator().manual_seed(team.SEED + 172)
    history = []
    for epoch in range(1, selected_epochs + 1):
        row = train_epoch(model, tail, optimizer, criterion, loader, blur_rng,
                          "full refit", epoch, selected_epochs)
        if row["images_trained"] != len(train_full):
            raise RuntimeError("A refit epoch did not process every training image.")
        schedule.step()
        history.append(row)
        print(row, flush=True)

    model.backbone.blocks[-1].load_state_dict(tail.block.state_dict())
    model.backbone.norm.load_state_dict(tail.norm.state_dict())
    model.crop_head.load_state_dict(tail.crop_head.state_dict())
    model.disease_head.load_state_dict(tail.disease_head.state_dict())
    sample = PlantDataset(train_full, image_transform, "imagenet")[0][0]
    verify_prefix(model, tail, sample.unsqueeze(0).to(DEVICE))
    model.cpu()
    refit = {"initialization": "original_pretrained_dinov2_fresh_heads",
             "data": "all_training_csv_rows", "images_per_epoch": len(train_full),
             "training_pairs": len(train_pairs), "epochs": selected_epochs, "cosine_t_max": EPOCHS}
    checkpoint_metadata = {"settings": settings, "recipe": RECIPE, "selection": development["selection"],
                           "gamma": development["gamma"], "seen_pairs": train_pairs.tolist(), "refit": refit}
    CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
    FINAL_DIR.mkdir(parents=True, exist_ok=True)
    pd.DataFrame(history).to_csv(FINAL_DIR / "refit_history.csv", index=False)
    pd.DataFrame([{**development["selection"], "recipe": RECIPE, "locked_gamma": development["gamma"]}]).to_csv(
        FINAL_DIR / "selected_model.csv", index=False)
    atomic_save(CHECKPOINT, {"state_dict": model.state_dict(), **checkpoint_metadata})
    locked = {**development, **checkpoint_metadata, "checkpoint_sha256": sha256_file(CHECKPOINT),
              "checkpoint_bytes": CHECKPOINT.stat().st_size}
    atomic_json(LOCK, locked)
    return locked


run = None if DEBUG else refit_full_train(development)
if run is not None:
    print(f"Full refit locked: {run['refit']['images_per_epoch']} images per epoch, "
          f"{run['refit']['epochs']} epochs.")

## 6. Official evaluation and save

The official shortlist adds the ten unseen concept tuples after the selection lock exists. Use only crop and disease IDs to build the shortlist. Read image names for inference, finish every prediction, then attach per-image targets by pair identity. The fourth official CSV column is never needed for pre-prediction ranking. The shared confusion-matrix labels are aligned afterwards.

See `results/approach2_finetune_a4_full_refit/<run_id>/official/pair_identity_order/official_comparison.csv` for raw and calibrated A4 results. `unseen_ranking_diagnostics.csv` tracks crop and disease head Top-1 and where a correct unseen-only Top-1 loses to a seen pair in the 333-way task. If the corrected Matched Selection report is available and its development settings match, `study_comparison.csv` includes B0–A4 and labels each run’s final training data. The full A4 weights are in `checkpoints/approach2_finetune_a4_full_refit/<run_id>/model.pt` (roughly 344 MB). The merged report is created after A4's own predictions and never enters selection.

In [ ]:
def official_vocabulary():
    # The final shortlist uses only concept identities and discards per-image row order.
    vocabulary = pd.read_csv(team.TEST_UNSEEN_CSV, header=None, usecols=[1, 2],
                             dtype=str, keep_default_na=False)
    vocabulary.columns = ["crop", "disease"]
    if not vocabulary.empty and not vocabulary.iloc[0]["crop"].strip().isdigit():
        vocabulary = vocabulary.iloc[1:]
    vocabulary = (vocabulary.astype("int64").drop_duplicates()
                  .sort_values(["crop", "disease"]))
    extra_pairs = torch.as_tensor(vocabulary.to_numpy(copy=True), dtype=torch.long)
    pairs = torch.cat((train_pairs, extra_pairs))
    if (len(extra_pairs) != team.NUM_UNSEEN_PAIRS or len(pairs.unique(dim=0)) != team.NUM_PAIRS
            or not set(extra_pairs[:, 0].tolist()) <= set(train_pairs[:, 0].tolist())
            or not set(extra_pairs[:, 1].tolist()) <= set(train_pairs[:, 1].tolist())):
        raise ValueError("Expected distinct official unseen pairs composed of training-derived concepts.")
    return pairs


def official_filenames(path):
    values = pd.read_csv(path, header=None, usecols=[0], dtype=str, keep_default_na=False).iloc[:, 0].tolist()
    header = {"image", "image_path", "image_name", "filename", "file_name", "file_path", "path"}
    if values and (values[0].strip().lower().replace(" ", "_") in header or
                   (len(values) > 1 and not Path(values[0]).suffix and Path(values[1]).suffix)):
        values = values[1:]
    names = [image_key(name) for name in values]
    if not names or len(set(names)) != len(names):
        raise ValueError(f"Expected unique image names in {path.name}.")
    return names


class OfficialImages(Dataset):
    def __init__(self, names):
        self.names = names

    def __len__(self):
        return len(self.names)

    def __getitem__(self, index):
        with Image.open(team.IMAGE_DIR / self.names[index]) as image:
            return image_transform(image.convert("RGB"))


def official_targets(csv_path, filenames, row_by_pair, *, seen):
    # This per-image projection is used after predictions. Ignore the official pair_label column.
    frame = pd.read_csv(csv_path, header=None, usecols=[0, 1, 2], dtype=str, keep_default_na=False)
    frame.columns = ["image", "crop", "disease"]
    if not frame.empty and not frame.iloc[0]["crop"].strip().isdigit():
        frame = frame.iloc[1:]
    if frame.image.map(image_key).tolist() != filenames:
        raise ValueError("Official targets and predictions have different image orders.")
    keys = frame[["crop", "disease"]].astype("int64").itertuples(index=False, name=None)
    try:
        labels = torch.tensor([row_by_pair[key] for key in keys], dtype=torch.long)
    except KeyError as error:
        raise ValueError(f"Official ground-truth pair absent from the frozen candidate table: {error}") from error
    if not ((labels < team.NUM_SEEN_PAIRS) == seen).all():
        raise ValueError("Official test split does not match seen membership.")
    return labels


def top_predictions(scores, offset=0):
    return {k: scores.topk(k, dim=1).indices + offset for k in (1, 5)}


def prediction_accuracy(predictions, labels):
    return round(100 * predictions.eq(labels[:, None]).any(1).float().mean().item(), 2)


@torch.inference_mode()
def predict_official(model, pairs, filenames, split, perturb=None):
    loader = DataLoader(OfficialImages(filenames), batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    mean, std = (torch.tensor(v).view(1, 3, 1, 1) for v in team.NORM_STATS["imagenet"])
    parts, concepts = [], []
    for step, images in enumerate(loader, 1):
        if perturb is not None:
            images = (perturb(images * std + mean) - mean) / std
        crop_logits, disease_logits = model(images.to(DEVICE))
        scores = pair_scores(crop_logits, disease_logits, pairs).cpu()
        if perturb is None:
            concepts.append(torch.stack((crop_logits.argmax(1), disease_logits.argmax(1)), dim=1).cpu())
        if scores.shape != (len(images), len(pairs)) or not torch.isfinite(scores).all():
            raise ValueError("Unexpected official candidate predictions.")
        parts.append(scores)
        if step == 1 or step % 50 == 0 or step == len(loader):
            print(f"{split}: batch {step}/{len(loader)}", flush=True)
    return torch.cat(parts), torch.cat(concepts) if concepts else None


def official_evaluation():
    if DEBUG or run is None:
        raise RuntimeError("Complete a full A4 development run first.")
    if read_final_lock() != run:
        raise RuntimeError("Checkpoint or scoring rule changed.")
    OFFICIAL_DIR.mkdir(parents=True, exist_ok=True)
    report_path = OFFICIAL_DIR / "official_report.json"
    if report_path.is_file():
        existing = json.loads(report_path.read_text(encoding="utf-8"))
        if existing["lock"] != run:
            raise RuntimeError("An existing report belongs to a different lock.")
        return existing

    saved = torch.load(CHECKPOINT, map_location="cpu", weights_only=True)
    model = MultiTaskNet(pretrained=False)
    model.load_state_dict(saved["state_dict"], strict=True)
    model.to(DEVICE).eval()
    pairs = official_vocabulary()
    paths = (team.TEST_SEEN_CSV, team.TEST_UNSEEN_CSV)
    filenames = tuple(official_filenames(path) for path in paths)
    if set(filenames[0]) & set(filenames[1]) or set(train_full.image) & (set(filenames[0]) | set(filenames[1])):
        raise ValueError("Train and official split filenames overlap.")
    boundary = len(train_pairs)
    membership = torch.arange(len(pairs)) < boundary
    clean, heads = zip(*(predict_official(model, pairs, names, split)
                         for names, split in zip(filenames, ("seen", "unseen"))))
    perturbed = {}
    with torch.random.fork_rng(devices=[]):
        torch.random.default_generator.manual_seed(team.SEED)
        for condition, transform in PERTURBATIONS.items():
            perturbed[condition], _ = predict_official(model, pairs, filenames[0], condition, transform)

    predictions = {}
    for rule, gamma in (("raw", 0.0), ("calibrated", run["gamma"])):
        seen_scores, unseen_scores = (scores - gamma * membership for scores in clean)
        predictions[rule] = {
            "gamma": gamma,
            "seen": top_predictions(seen_scores),
            "unseen": top_predictions(unseen_scores),
            "seen_only": top_predictions(seen_scores[:, :boundary]),
            "unseen_only": top_predictions(unseen_scores[:, boundary:], offset=boundary),
            "robust": {name: (scores - gamma * membership).topk(1, dim=1).indices
                       for name, scores in perturbed.items()},
        }

    # Only after all candidate predictions are frozen, attach per-image truth.
    rows = {tuple(pair): i for i, pair in enumerate(pairs.tolist())}
    y_seen = official_targets(paths[0], filenames[0], rows, seen=True)
    y_unseen = official_targets(paths[1], filenames[1], rows, seen=False)
    y_all = torch.cat((y_seen, y_unseen))
    canonical = {tuple(pair): row for row, pair in enumerate(get_pairs(final_evaluation=True).tolist())}
    if canonical.keys() != rows.keys():
        raise ValueError("Candidate set differs from the shared test vocabulary.")
    to_canonical = torch.tensor([canonical[tuple(pair)] for pair in pairs.tolist()])
    if (to_canonical[y_seen] >= boundary).any():
        raise ValueError("The shared confusion table expects seen target rows.")

    comparison, restricted, robustness, ranking_diagnostics = [], [], [], []
    unseen_head_hits = heads[1].eq(pairs[y_unseen])
    unseen_crop_top1, unseen_disease_top1 = (unseen_head_hits.float().mean(dim=0) * 100).tolist()
    for rule, pred in predictions.items():
        row = {"recipe": RECIPE, "scoring": rule, "gamma": pred["gamma"],
               "epoch": run["selection"]["epoch"], "candidates": len(pairs)}
        for k in (1, 5):
            row[f"seen_top{k}"] = prediction_accuracy(pred["seen"][k], y_seen)
            row[f"unseen_top{k}"] = prediction_accuracy(pred["unseen"][k], y_unseen)
            row[f"overall_top{k}"] = prediction_accuracy(torch.cat((pred["seen"][k], pred["unseen"][k])), y_all)
        row["harmonic_top1"] = harmonic_mean(row["seen_top1"], row["unseen_top1"])
        comparison.append(row)
        correct_within_unseen = pred["unseen_only"][1].squeeze(1).eq(y_unseen)
        joint_unseen = pred["unseen"][1].squeeze(1)
        predicted_seen = joint_unseen < boundary
        margin_scores = clean[1] - pred["gamma"] * membership
        target_scores = margin_scores.gather(1, y_unseen[:, None]).squeeze(1)
        best_seen = margin_scores[:, :boundary].amax(dim=1)
        ranking_diagnostics.append({
            "scoring": rule,
            "unseen_images": len(y_unseen),
            "crop_head_top1": round(unseen_crop_top1, 2),
            "disease_head_top1": round(unseen_disease_top1, 2),
            "joint_correct": int(joint_unseen.eq(y_unseen).sum()),
            "unseen_only_correct": int(correct_within_unseen.sum()),
            "unseen_predicted_seen": int(predicted_seen.sum()),
            "correct_within_unseen_but_blocked_by_seen": int((correct_within_unseen & predicted_seen).sum()),
            "median_target_minus_top_seen_score": float((target_scores - best_seen).median()),
        })
        for split, target, count in (("seen_only", y_seen, boundary), ("unseen_only", y_unseen, len(pairs) - boundary)):
            restricted.append({"recipe": RECIPE, "scoring": rule, "split": split, "candidates": count,
                               **{f"top{k}": prediction_accuracy(pred[split][k], target) for k in (1, 5)}})
        for condition, indices in pred["robust"].items():
            accuracy = prediction_accuracy(indices, y_seen)
            robustness.append({"recipe": RECIPE, "scoring": rule, "condition": condition,
                               "seen_top1": accuracy, "drop_pp": round(row["seen_top1"] - accuracy, 2)})
        votes = F.one_hot(to_canonical[pred["seen"][1].squeeze(1)], num_classes=len(pairs)).float()
        stem = (OFFICIAL_DIR / f"{RECIPE}_{rule}").relative_to(team.RESULTS_DIR).as_posix()
        save_confusion(votes, to_canonical[y_seen], stem, final_evaluation=True)

    pd.DataFrame(comparison).to_csv(OFFICIAL_DIR / "official_comparison.csv", index=False)
    pd.DataFrame(restricted).to_csv(OFFICIAL_DIR / "restricted_candidates.csv", index=False)
    pd.DataFrame(robustness).to_csv(OFFICIAL_DIR / "official_robustness.csv", index=False)
    pd.DataFrame(ranking_diagnostics).to_csv(OFFICIAL_DIR / "unseen_ranking_diagnostics.csv", index=False)
    report = {"lock": run, "pair_order": "train_then_concept_ids", "candidate_pairs": pairs.tolist(),
              "test_counts": {"seen": len(y_seen), "unseen": len(y_unseen)},
              "comparison": comparison, "restricted_candidates": restricted, "robustness": robustness,
              "unseen_ranking_diagnostics": ranking_diagnostics}
    atomic_json(report_path, report)
    return report


def attach_a3_comparison(report):
    if not REFERENCE_REPORT.is_file():
        print("A4 metrics saved. Run the corrected Matched Selection evaluation to add the A3 comparison.")
        return
    reference = json.loads(REFERENCE_REPORT.read_text(encoding="utf-8"))
    baseline_settings = reference["lock"]["settings"]
    a3_settings = baseline_settings.get("recipes", {}).get("A3_Add_Blur")
    matched_keys = ("backbone", "seed", "image_size", "resize_size", "val_fraction",
                    "batch_size", "epochs", "weight_decay", "optimizer", "schedule", "label_smoothing",
                    "temperature", "blur_probability", "blur_sigma", "train_csv_sha256",
                    "calibration_grid", "reference_commit")
    if (reference.get("pair_order") != report["pair_order"]
            or reference["candidate_pairs"] != report["candidate_pairs"]
            or reference["test_counts"] != report["test_counts"]
            or any(baseline_settings.get(key) != settings[key] for key in matched_keys)
            or baseline_settings["lr"] != HEAD_LR
            or a3_settings != [DECOR_WEIGHT, SUPCON_WEIGHT, True]
            or any(reference["lock"][key] != run[key] for key in
                   ("selection_tune_pair_rows", "calibration_check_pair_rows"))):
        print("Existing A3 report does not match the A4 protocol. Keeping the A4 results separate.")
        return
    reference_data = reference["lock"].get("refit", {}).get("data", "team_train_split")
    comparison = pd.concat([
        pd.DataFrame(reference["comparison"]).assign(final_training_data=reference_data),
        pd.DataFrame(report["comparison"]).assign(final_training_data=report["lock"]["refit"]["data"]),
    ], ignore_index=True)
    unseen_only = pd.DataFrame([*reference["restricted_candidates"], *report["restricted_candidates"]])
    unseen_only = unseen_only.loc[unseen_only.split.eq("unseen_only"), ["recipe", "scoring", "top1"]]
    comparison = comparison.merge(unseen_only.rename(columns={"top1": "unseen_only_top1"}),
                                  how="left", on=["recipe", "scoring"], validate="1:1")
    comparison.to_csv(OFFICIAL_DIR / "study_comparison.csv", index=False)
    primary = comparison[comparison.scoring.eq("raw") & comparison.recipe.isin(["A3_Add_Blur", RECIPE])]
    display(primary[["recipe", "final_training_data", "epoch", "seen_top1", "unseen_top1",
                     "unseen_only_top1", "harmonic_top1"]])


if DEBUG:
    print(f"Debug development logs: {DEVELOPMENT_DIR.resolve()}")
else:
    report = official_evaluation()
    display(pd.DataFrame(report["comparison"]))
    attach_a3_comparison(report)
    print(f"Full A4 checkpoint: {CHECKPOINT.resolve()}")
    print(f"Official reports: {OFFICIAL_DIR.resolve()}")